# NB07 — Train SAC (Soft Actor-Critic)

Train a SAC agent on `UnitreeG1DishWipe-v1` using Stable-Baselines3.
SAC is an off-policy algorithm that maximises expected return plus an entropy
bonus, often achieving better sample efficiency than PPO.


## Objective

1. Create vectorised environment (same setup as NB06).
2. Configure SAC with automatic entropy tuning.
3. Train for the same `TOTAL_ENV_STEPS` as PPO (fair comparison).
4. Save model, learning curve, and log to MLflow.


## Environment

| Notebook | Goal | Required HW | Min CPU | Min RAM | GPU VRAM | Notes |
|---|---|---|---:|---:|---:|---|
| NB07 | Train SAC | GPU (practical) | 8 cores | 16 GB | 12-24 GB | RunPod 4090+ |


In [ ]:
import sys, os, platform, json
print(f"Python: {sys.version}"); print(f"OS: {platform.platform()}")
import numpy as np; print(f"NumPy: {np.__version__}")
import torch; print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA: not available (CPU mode)")
import gymnasium as gym; print(f"Gymnasium: {gym.__version__}")
import stable_baselines3 as sb3; print(f"SB3: {sb3.__version__}")


## Imports


In [ ]:
import json, time
import numpy as np
import torch
import gymnasium as gym
from pathlib import Path
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed

# Ensure project root is on sys.path so `src.envs` can be imported
PROJECT_ROOT = Path(os.getcwd()).resolve()
if (PROJECT_ROOT / "src").is_dir():
    pass  # already at project root
elif (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError(f"Cannot find src/ from {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # normalize cwd for artifact paths

from src.envs.dishwipe_env import UnitreeG1DishWipeEnv  # registers env


## Config


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IS_GPU = DEVICE == "cuda"

CFG = dict(
    seed=42,
    env_id="UnitreeG1DishWipe-v1",
    control_mode="pd_joint_delta_pos",
    obs_mode="state",
    # ── Training budget (same as PPO) ──
    total_env_steps=500_000 if IS_GPU else 20_000,
    n_envs=1,  # SAC is off-policy, typically 1 env
    # ── SAC hyperparameters ──
    learning_rate=3e-4,
    buffer_size=1_000_000 if IS_GPU else 50_000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    ent_coef="auto",       # automatic entropy tuning
    target_entropy="auto",
    learning_starts=1000,
    train_freq=1,
    gradient_steps=1,
    # ── Network ──
    net_arch=[256, 256],
    # ── Evaluation ──
    eval_episodes=10,
    # ── Device ──
    device=DEVICE,
)
SEED = CFG["seed"]
artifact_dir = Path("artifacts/NB07")
artifact_dir.mkdir(parents=True, exist_ok=True)
print("Config:", json.dumps(CFG, indent=2, default=str))


## Reproducibility


In [ ]:
import random; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
set_random_seed(SEED)
print(f"✅ Seeds set to {SEED}")


## Implementation Steps

### Step 1 — Create environment


In [ ]:
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

env = gym.make(
    CFG["env_id"], obs_mode=CFG["obs_mode"],
    control_mode=CFG["control_mode"], num_envs=CFG["n_envs"], render_mode=None,
)
env = CPUGymWrapper(env)
print(f"Train env: obs={env.observation_space.shape}, act={env.action_space.shape}")

eval_env = gym.make(
    CFG["env_id"], obs_mode=CFG["obs_mode"],
    control_mode=CFG["control_mode"], num_envs=1, render_mode=None,
)
eval_env = CPUGymWrapper(eval_env)
print(f"Eval env: obs={eval_env.observation_space.shape}")


### Step 2 — Configure & create SAC


In [ ]:
model = SAC(
    "MlpPolicy",
    env,
    learning_rate=CFG["learning_rate"],
    buffer_size=CFG["buffer_size"],
    batch_size=CFG["batch_size"],
    tau=CFG["tau"],
    gamma=CFG["gamma"],
    ent_coef=CFG["ent_coef"],
    target_entropy=CFG["target_entropy"],
    learning_starts=CFG["learning_starts"],
    train_freq=CFG["train_freq"],
    gradient_steps=CFG["gradient_steps"],
    policy_kwargs=dict(net_arch=CFG["net_arch"]),
    verbose=1,
    seed=SEED,
    device=DEVICE,
)
print(f"\nSAC model created on {DEVICE}")
print(f"Policy: {model.policy}")


### Step 3 — Train


In [ ]:
class TrainLogCallback(BaseCallback):
    def __init__(self, log_freq=1000, verbose=0):
        super().__init__(verbose)
        self.log_freq = log_freq
        self.rewards = []

    def _on_step(self):
        if self.n_calls % self.log_freq == 0:
            if len(self.model.ep_info_buffer) > 0:
                mean_rew = np.mean([ep["r"] for ep in self.model.ep_info_buffer])
                self.rewards.append((self.num_timesteps, mean_rew))
                if self.verbose:
                    print(f"  Step {self.num_timesteps}: mean_reward={mean_rew:.3f}")
        return True

log_callback = TrainLogCallback(log_freq=2000, verbose=1)

print(f"Training SAC for {CFG['total_env_steps']} steps...")
t0 = time.time()
model.learn(
    total_timesteps=CFG["total_env_steps"],
    callback=[log_callback],
    progress_bar=True,
)
train_time = time.time() - t0
print(f"\n✅ Training complete in {train_time:.1f}s")


### Step 4 — Save model & evaluate


In [ ]:
model_path = artifact_dir / "sac_model"
model.save(str(model_path))
print(f"✅ Model saved: {model_path}.zip")

# Deterministic evaluation (use mean action)
eval_rewards = []
eval_cleaned = []
for ep in range(CFG["eval_episodes"]):
    obs, info = eval_env.reset(seed=SEED + 2000 + ep)
    total_rew = 0.0
    for step in range(1000):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_rew += reward.item() if hasattr(reward, "item") else float(reward)
        if terminated.any() if hasattr(terminated, "any") else terminated:
            break
    ratio = info.get("cleaned_ratio", torch.tensor([0.0]))
    eval_rewards.append(total_rew)
    eval_cleaned.append(ratio.item() if hasattr(ratio, "item") else float(ratio))

print(f"\nEvaluation ({CFG['eval_episodes']} episodes):")
print(f"  Reward  : {np.mean(eval_rewards):.3f} ± {np.std(eval_rewards):.3f}")
print(f"  Cleaned : {np.mean(eval_cleaned):.4f} ± {np.std(eval_cleaned):.4f}")


### Step 5 — Learning curve


In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

if log_callback.rewards:
    steps, rewards = zip(*log_callback.rewards)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, rewards, linewidth=1.5, color="tab:orange")
    ax.set_xlabel("Environment Steps")
    ax.set_ylabel("Mean Episode Reward")
    ax.set_title("NB07 — SAC Learning Curve (UnitreeG1DishWipe-v1)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    curve_path = artifact_dir / "learning_curve.png"
    fig.savefig(str(curve_path), dpi=120, bbox_inches="tight")
    plt.close("all")
    print(f"✅ Saved: {curve_path}")
else:
    print("⚠️ No training logs for plotting.")


### Step 6 — Artifacts & MLflow


In [ ]:
eval_results = dict(
    mean_eval_reward=float(np.mean(eval_rewards)),
    std_eval_reward=float(np.std(eval_rewards)),
    mean_eval_cleaned=float(np.mean(eval_cleaned)),
    train_time_seconds=train_time,
)
with open(artifact_dir / "eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
with open(artifact_dir / "nb07_config.json", "w") as f:
    json.dump(CFG, f, indent=2, default=str)

try:
    import mlflow; from dotenv import load_dotenv; load_dotenv(".env.local")
    uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if uri:
        mlflow.set_tracking_uri(uri)
        mlflow.set_experiment("dishwipe_unitree_g1")
        with mlflow.start_run(run_name="NB07_SAC"):
            mlflow.log_params({k: v for k, v in CFG.items()
                              if not isinstance(v, (list, dict))})
            mlflow.log_metric("eval_reward_mean", eval_results["mean_eval_reward"])
            mlflow.log_metric("eval_cleaned_mean", eval_results["mean_eval_cleaned"])
            mlflow.log_metric("train_time_s", train_time)
            mlflow.log_artifacts(str(artifact_dir), artifact_path="NB07")
        print("✅ MLflow run logged.")
except Exception as e:
    print(f"⚠️ MLflow: {e}")


## Results

- SAC trained on UnitreeG1DishWipe-v1 with automatic entropy tuning
- Same TOTAL_ENV_STEPS as PPO for fair comparison
- Model saved as `artifacts/NB07/sac_model.zip`


## Artifacts

| File | Description |
|------|-------------|
| `artifacts/NB07/sac_model.zip` | Trained SAC model |
| `artifacts/NB07/learning_curve.png` | Training reward curve |
| `artifacts/NB07/eval_results.json` | Evaluation metrics |


## Cleanup


In [ ]:
env.close()
eval_env.close()
print('✅ NB07 complete.')


## References

- Haarnoja et al. (2018) — Soft Actor-Critic
- SB3 SAC: https://stable-baselines3.readthedocs.io/en/master/modules/sac.html
- Maximum entropy RL: policy entropy bonus for exploration
